In [13]:
# Generate strict 30s DAS images (3 consecutive 10s files) from GC/Data/20250702
import os, glob, math, h5py, numpy as np, matplotlib.pyplot as plt
from datetime import datetime, timedelta

# ---- Parameters ----
INPUT_DIR = "../../raw_data/GC_Data/20250702"   # directory containing HHMMSS.hdf5 files
OUTPUT_DIR = "unseen_data"       # save location (adjust if needed)
FS_ASSUMED = 625.0                      # Hz
DX_ASSUMED_M = 1.0213001907746815       # meters per channel
DIST_MIN_KM = 4                         # lower distance bound
DIST_MAX_KM = 14                        # upper distance bound
VMAX = 700                              # clip / scale max
IMG_H, IMG_W = 512, 1024                # output resolution
CMAP = "jet"
DPI = 600
START_FROM_TIME = "124226"             # earliest HHMMSS to include

os.makedirs(OUTPUT_DIR, exist_ok=True)

def parse_time(path):
    base = os.path.splitext(os.path.basename(path))[0]
    try:
        return datetime.strptime(base, "%H%M%S")
    except Exception:
        return None

def is_next_10s(t1, t2):
    return t1 is not None and t2 is not None and t1 + timedelta(seconds=10) == t2

# Collect files
all_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.hdf5")))
if not all_files:
    raise FileNotFoundError(f"No HDF5 files found in {INPUT_DIR}")

# Filter start
try:
    start_dt = datetime.strptime(START_FROM_TIME, "%H%M%S")
    all_files = [f for f in all_files if (parse_time(f) or datetime.min) >= start_dt]
except Exception:
    pass

print(f"Found {len(all_files)} candidate 10s files from {INPUT_DIR} (start >= {START_FROM_TIME})")

# Build non-overlapping triplets (strict continuity)
triplets = []
i = 0
while i + 2 < len(all_files):
    f1, f2, f3 = all_files[i], all_files[i+1], all_files[i+2]
    t1, t2, t3 = parse_time(f1), parse_time(f2), parse_time(f3)
    if is_next_10s(t1, t2) and is_next_10s(t2, t3):
        triplets.append((f1, f2, f3, t1))
        i += 3  # advance by 3 to make next window start 30s later
    else:
        i += 1  # slide until continuity achieved

print(f"Identified {len(triplets)} strict 30s windows")

saved = 0
for (p1, p2, p3, t_start) in triplets:
    bases = [os.path.splitext(os.path.basename(p))[0] for p in (p1, p2, p3)]
    try:
        slices = []
        for p in (p1, p2, p3):
            with h5py.File(p, 'r') as f:
                if 'strainrate' in f:
                    dset = f['strainrate']
                elif 'data' in f:
                    dset = f['data']
                else:
                    dset = None
                if dset is None or dset.ndim != 2:
                    print(f"Skipping triplet {bases}: bad array in {os.path.basename(p)}")
                    slices = []
                    break
                T, C = dset.shape
                if T / FS_ASSUMED < 9.0:
                    print(f"Skipping triplet {bases}: {os.path.basename(p)} <10s")
                    slices = []
                    break
                distances_km = (np.arange(C) * DX_ASSUMED_M) / 1000.0
                dmin_eff = max(DIST_MIN_KM, float(distances_km[0]))
                dmax_eff = min(DIST_MAX_KM, float(distances_km[-1]))
                if dmax_eff <= dmin_eff:
                    print(f"Skipping triplet {bases}: invalid distance bounds")
                    slices = []
                    break
                ch_lo = int(math.ceil((dmin_eff * 1000.0) / DX_ASSUMED_M))
                ch_hi = int(math.floor((dmax_eff * 1000.0) / DX_ASSUMED_M)) + 1
                ch_lo = max(0, min(ch_lo, C-1))
                ch_hi = max(ch_lo+1, min(ch_hi, C))
                if ch_hi - ch_lo < 2:
                    print(f"Skipping triplet {bases}: insufficient channels")
                    slices = []
                    break
                window = dset[:, ch_lo:ch_hi].astype(np.float32)
                slices.append(window)
        if len(slices) != 3:
            continue
        stitched = np.concatenate(slices, axis=0)
        stitched = np.clip(stitched, 0.0, VMAX) / VMAX

        fig = plt.figure(figsize=(IMG_W / DPI, IMG_H / DPI), dpi=DPI)
        ax = fig.add_axes([0, 0, 1, 1])
        ax.axis('on')
        ax.imshow(stitched, aspect='auto', cmap=CMAP, vmin=0.0, vmax=1.0, interpolation='nearest')
        start_str = t_start.strftime('%H%M%S') if t_start else bases[0]
        out_name = f"{start_str}_T30s.png"
        out_path = os.path.join(OUTPUT_DIR, out_name)
        plt.xlabel("Distance (km)", fontsize=16)
        plt.ylabel("Time (UTC)", fontsize=16)
        fig.savefig(out_path, dpi=DPI, bbox_inches='tight')
        plt.close(fig)
        saved += 1
        print(f"Saved {out_name}: triplet {bases} distance {DIST_MIN_KM}-{DIST_MAX_KM} km cols {stitched.shape[1]}")
    except Exception as e:
        print(f"Error processing triplet {bases}: {e}")

print(f"Total saved 60s images: {saved}")

Found 266 candidate 10s files from ../../raw_data/GC_Data/20250702 (start >= 124226)
Identified 88 strict 30s windows
Saved 124226_T30s.png: triplet ['124226', '124236', '124246'] distance 4-14 km cols 9792
Saved 124226_T30s.png: triplet ['124226', '124236', '124246'] distance 4-14 km cols 9792
Saved 124256_T30s.png: triplet ['124256', '124306', '124316'] distance 4-14 km cols 9792
Saved 124256_T30s.png: triplet ['124256', '124306', '124316'] distance 4-14 km cols 9792
Saved 124326_T30s.png: triplet ['124326', '124336', '124346'] distance 4-14 km cols 9792
Saved 124326_T30s.png: triplet ['124326', '124336', '124346'] distance 4-14 km cols 9792


KeyboardInterrupt: 